In [2]:
!pip install -q transformers peft trl accelerate
!pip install -q "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 11.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incom

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import pandas as pd

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [4]:
test_prompts = [
    "A store sells apples for $0.75 each and oranges for $1.25 each. If Sarah buys 4 apples and 3 oranges, how much does she spend in total?",
    "A train travels at 60 mph for 2.5 hours, then at 80 mph for 1.5 hours. What is the total distance traveled?",
    "A class has 32 students. 3/8 of them play football and 1/4 of them play basketball. How many students play neither sport?",
    "John earns $15 per hour and works 8 hours a day, 5 days a week. He saves 20% of his weekly earnings. How much does he save per week?",
    "A rectangle has a perimeter of 54 cm. If its length is twice its width, what is the area of the rectangle?",
    "A tank is 1/3 full of water. After adding 24 liters, it becomes 3/4 full. What is the total capacity of the tank?",
    "A shopkeeper bought 50 items for $200 and sold them all for $270. What is the profit percentage?",
    "Two friends split a restaurant bill. The bill was $84 before a 15% tip. They also have a coupon for $10 off the total after tip. How much does each person pay?",
    "A car depreciates in value by 15% each year. If it was worth $20,000 when new, what is it worth after 2 years?",
    "A pipe can fill a tank in 6 hours and another pipe can fill the same tank in 4 hours. If both pipes are opened together, how long will it take to fill the tank?",
]


trial_ids = [1, 2, 3, 4, 5]
all_outputs = []

for trial_id in trial_ids:
    adapter_path = f"/kaggle/input/datasets/kashishanilkumar92/sft-trial-{trial_id}"
    print(f"\nLoading Trial {trial_id} from {adapter_path}...")

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map={"": 0},
        trust_remote_code=True
    )
    peft_model = PeftModel.from_pretrained(base_model, adapter_path)
    peft_model.eval()

    trial_responses = []
    for i, prompt in enumerate(test_prompts):
        messages = [
            {"role": "system", "content": "You are a helpful reasoning assistant."},
            {"role": "user", "content": prompt}
        ]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

        with torch.no_grad():
            output = peft_model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                repetition_penalty=1.3,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        response = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        trial_responses.append(response)
        print(f"  Q{i+1} done")

    all_outputs.append({"trial_id": trial_id, "responses": trial_responses})

    del base_model, peft_model
    torch.cuda.empty_cache()

print("\nInference complete!")


Loading Trial 1 from /kaggle/input/datasets/kashishanilkumar92/sft-trial-1...


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Q1 done
  Q2 done
  Q3 done
  Q4 done
  Q5 done
  Q6 done
  Q7 done
  Q8 done
  Q9 done
  Q10 done

Loading Trial 5 from /kaggle/input/datasets/kashishanilkumar92/sft-trial-5...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Q1 done
  Q2 done
  Q3 done
  Q4 done
  Q5 done
  Q6 done
  Q7 done
  Q8 done
  Q9 done
  Q10 done

Inference complete!


In [6]:
rows = []
for entry in all_outputs:
    for i, (prompt, response) in enumerate(zip(test_prompts, entry["responses"])):
        rows.append({
            "trial_id": entry["trial_id"],
            "question_id": i + 1,
            "prompt": prompt,
            "model_response": response
        })

responses_df = pd.DataFrame(rows)
responses_df.to_csv("/kaggle/working/sft_all_trial_responses.csv", index=False)
print("Saved! Hand sft_all_trial_responses.csv to Member 1.")
print(responses_df)

Saved! Hand sft_all_trial_responses.csv to Member 1.
    trial_id  question_id                                             prompt  \
0          1            1  A store sells apples for $0.75 each and orange...   
1          1            2  A train travels at 60 mph for 2.5 hours, then ...   
2          1            3  A class has 32 students. 3/8 of them play foot...   
3          1            4  John earns $15 per hour and works 8 hours a da...   
4          1            5  A rectangle has a perimeter of 54 cm. If its l...   
5          1            6  A tank is 1/3 full of water. After adding 24 l...   
6          1            7  A shopkeeper bought 50 items for $200 and sold...   
7          1            8  Two friends split a restaurant bill. The bill ...   
8          1            9  A car depreciates in value by 15% each year. I...   
9          1           10  A pipe can fill a tank in 6 hours and another ...   
10         2            1  A store sells apples for $0.75 each and 

In [5]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/kashishanilkumar92/sft-trial-3/adapter_model.safetensors
/kaggle/input/datasets/kashishanilkumar92/sft-trial-3/training_args.bin
/kaggle/input/datasets/kashishanilkumar92/sft-trial-3/adapter_config.json
/kaggle/input/datasets/kashishanilkumar92/sft-trial-3/tokenizer.json
/kaggle/input/datasets/kashishanilkumar92/sft-trial-3/tokenizer_config.json
/kaggle/input/datasets/kashishanilkumar92/sft-trial-3/chat_template.jinja
/kaggle/input/datasets/kashishanilkumar92/sft-trial-4/adapter_model.safetensors
/kaggle/input/datasets/kashishanilkumar92/sft-trial-4/training_args.bin
/kaggle/input/datasets/kashishanilkumar92/sft-trial-4/adapter_config.json
/kaggle/input/datasets/kashishanilkumar92/sft-trial-4/tokenizer.json
/kaggle/input/datasets/kashishanilkumar92/sft-trial-4/tokenizer_config.json
/kaggle/input/datasets/kashishanilkumar92/sft-trial-4/chat_template.jinja
/kaggle/input/datasets/kashishanilkumar92/sft-trial-2/adapter_model.safetensors
/kaggle/input/datasets/kashish